In [ ]:
import os
import glob
import re
import numpy as np
import xarray as xr

era5_folder = "/work5/ERA5_daily_landgrid_only/global"

ai_file = os.path.join(
    era5_folder,
    "ERA5_aridity_index_percentile_1981_2000.nc"
)

latent_heat = 2.45e6  # J kg-1

# ============================================================
# Select monthly files from 1981–2000
# ============================================================
all_files = sorted(glob.glob(
    os.path.join(
        era5_folder,
        "compressed_sfc_fc_daily_*.nc4"
    )
))

era5_files = []

for file in all_files:
    match = re.search(r"daily_(\d{4})(\d{2})\.nc4$", file)

    if match:
        year = int(match.group(1))

        if 1981 <= year <= 2000:
            era5_files.append(file)

print("Number of monthly files:", len(era5_files))

assert len(era5_files) == 20 * 12, (
    f"Expected 240 files, but found {len(era5_files)}"
)

# ============================================================
# Read coordinates
# ============================================================
with xr.open_dataset(era5_files[0]) as ds:
    lat_1d = ds["lat"].values.astype(np.float64)
    lon_1d = ds["lon"].values.astype(np.float64)

n_grid = len(lat_1d)

radiation_sum = np.zeros(n_grid, dtype=np.float64)
precipitation_sum = np.zeros(n_grid, dtype=np.float64)
valid_count = np.zeros(n_grid, dtype=np.int64)

# ============================================================
# Accumulate daily radiation and precipitation
# ============================================================
for i, file in enumerate(era5_files, start=1):

    with xr.open_dataset(file) as ds:
        net_radiation = (
            ds["msnswrf"].values.astype(np.float64) +
            ds["msnlwrf"].values.astype(np.float64)
        )

        precipitation = (
            ds["mtpr"].values.astype(np.float64)
        )

    # Require both variables to be available
    valid = (
        np.isfinite(net_radiation) &
        np.isfinite(precipitation)
    )

    # Sum across time while retaining the lgrid dimension
    radiation_sum += np.where(
        valid,
        net_radiation,
        0.0
    ).sum(axis=0)

    precipitation_sum += np.where(
        valid,
        precipitation,
        0.0
    ).sum(axis=0)

    valid_count += valid.sum(axis=0)

    if i % 12 == 0:
        print(f"Processed {i}/{len(era5_files)} files")

# ============================================================
# Calculate 1981–2000 climatological means
# ============================================================
mean_net_radiation = np.full(
    n_grid,
    np.nan,
    dtype=np.float64
)

mean_precipitation = np.full(
    n_grid,
    np.nan,
    dtype=np.float64
)

has_data = valid_count > 0

mean_net_radiation[has_data] = (
    radiation_sum[has_data] /
    valid_count[has_data]
)

mean_precipitation[has_data] = (
    precipitation_sum[has_data] /
    valid_count[has_data]
)

# Convert net radiation to equivalent evaporation rate:
# W m-2 / J kg-1 = kg m-2 s-1
available_energy_equivalent = (
    mean_net_radiation / latent_heat
)

# ============================================================
# Calculate aridity index
# ============================================================
aridity_index = np.full(
    n_grid,
    np.nan,
    dtype=np.float64
)

valid_ai = (
    np.isfinite(available_energy_equivalent) &
    np.isfinite(mean_precipitation) &
    (mean_net_radiation > 0) &
    (mean_precipitation > 0)
)

aridity_index[valid_ai] = (
    available_energy_equivalent[valid_ai] /
    mean_precipitation[valid_ai]
)

In [ ]:
# ============================================================
# Area-weighted global percentile
# ============================================================
def area_weighted_percentile_rank(
    values,
    latitudes
):
    percentile = np.full(
        values.shape,
        np.nan,
        dtype=np.float64
    )

    valid = (
        np.isfinite(values) &
        np.isfinite(latitudes)
    )

    values_valid = values[valid]

    weights_valid = np.cos(
        np.deg2rad(latitudes[valid])
    )

    order = np.argsort(values_valid)

    sorted_weights = weights_valid[order]

    cumulative_weight = (
        np.cumsum(sorted_weights) -
        0.5 * sorted_weights
    )

    sorted_percentile = (
        100.0 *
        cumulative_weight /
        np.sum(sorted_weights)
    )

    original_percentile = np.empty_like(
        sorted_percentile
    )

    original_percentile[order] = sorted_percentile
    percentile[valid] = original_percentile

    return percentile


aridity_percentile = area_weighted_percentile_rank(
    aridity_index,
    lat_1d
).astype(np.float32)

# ============================================================
# Save output
# ============================================================
ds_out = xr.Dataset(
    data_vars={
        "aridity_index": (
            "lgrid",
            aridity_index.astype(np.float32)
        ),
        "aridity_percentile": (
            "lgrid",
            aridity_percentile
        ),
        "mean_net_radiation": (
            "lgrid",
            mean_net_radiation.astype(np.float32)
        ),
        "mean_precipitation_rate": (
            "lgrid",
            mean_precipitation.astype(np.float32)
        )
    },
    coords={
        "lgrid": np.arange(n_grid),
        "lat": ("lgrid", lat_1d.astype(np.float32)),
        "lon": ("lgrid", lon_1d.astype(np.float32))
    },
    attrs={
        "period": "1981–2000",
        "aridity_index_definition": (
            "(net radiation / latent heat of vaporization) "
            "/ precipitation rate"
        ),
        "percentile_definition": (
            "Area-weighted global land percentile"
        )
    }
)

ds_out["aridity_index"].attrs["units"] = "1"
ds_out["aridity_percentile"].attrs["units"] = "%"
ds_out["mean_net_radiation"].attrs["units"] = "W m-2"
ds_out["mean_precipitation_rate"].attrs["units"] = (
    "kg m-2 s-1"
)

encoding = {
    variable: {
        "zlib": True,
        "complevel": 4,
        "dtype": "float32"
    }
    for variable in ds_out.data_vars
}
import os

ai_outdir = "/work7/IPCS_extreme/Plot_main_text_Figure/outcomes"

os.makedirs(ai_outdir, exist_ok=True)

ai_file = os.path.join(
    ai_outdir,
    "ERA5_aridity_index_percentile_1981_2000.nc"
)
ds_out.to_netcdf(
    ai_file,
    encoding=encoding
)

print(f"Saved: {ai_file}")